# Progetto Data Mining 1 — A.A. 2026/2027
## Modulo 1: Data Understanding & Preparation
### Dataset: *Seller-Level Vehicle Dataset*

Questo notebook contiene l'analisi del **Modulo 1 (Data Understanding & Preparation)** per il progetto di Data Mining 1 (Laurea Magistrale in Data Science / Informatica Umanistica, Università di Pisa).

**Obiettivi e struttura del Modulo 1:**
1. **Data Semantics & Initial Inspection (Punti 1.1 e 1.2)**: Comprensione del significato applicativo di ciascuna colonna, classificazione secondo le scale statistiche di misura (Nominale, Ordinale, Intervallo, Ratio) e assegnazione del ruolo nel mining (ID, Predittore, Target).
2. **Data Quality & Variable Transformation (Punto 1.3)**: Identificazione e correzione di anomalie sintattiche/semantiche (es. formati errati), gestione trasparente dei missing values, rilevamento outlier e studio di trasformazioni.
3. **Distribution & Statistics (Punto 1.4)**: Studio quantitativo univariato e bivariato delle distribuzioni rispetto al target.
4. **Correlations & Feature Engineering (Punto 1.5)**: Analisi della collinearità, selezione/riduzione di attributi e costruzione del dataset finale preparato (*seller_prepared.csv*).

---
## 1. Setup dell'Ambiente e Librerie

In questa prima cella importiamo le librerie Python fondamentali per l'analisi dati e la statistica.

* **`pandas`**: La libreria standard per manipolare dati in formato tabellare (**DataFrame**). Permette di caricare file CSV/Excel, filtrare, aggregare, calcolare statistiche ed esplorare le colonne.
* **`numpy`**: Libreria di calcolo numerico vettoriale ad alte prestazioni. Gestisce array matematici e funzioni numeriche (come logaritmi, radici, e il valore speciale `np.nan` per indicare valori nulli).
* **`matplotlib.pyplot`**: Libreria di base per la visualizzazione grafica 2D in Python.
* **`seaborn`**: Libreria per la visualizzazione statistica costruita sopra matplotlib. Fornisce grafici eleganti come istogrammi con stima di densità continua (KDE), boxplot e heatmap di correlazione.

In [ ]:
# ==============================================================================
# IMPORT DELLE LIBRERIE FONDAMENTALI
# ==============================================================================

import pandas as pd           # Manipolazione e analisi di dati tabellari
import numpy as np            # Calcolo numerico, array e funzioni matematiche
import matplotlib.pyplot as plt # Creazione di grafici e figure 2D
import seaborn as sns         # Grafici statistici avanzati
import os                     # Gestione percorsi di sistema
from IPython.display import display # Visualizzazione ricca di tabelle e output

# sns.set_theme: Imposta lo stile visivo predefinito di Seaborn.
# 'whitegrid' aggiunge un reticolo grigio chiaro su sfondo bianco, facilitando la lettura dei valori numerici.
sns.set_theme(style="whitegrid")

# %matplotlib inline: Magic command specifico di IPython/Jupyter Notebook.
# Forza la visualizzazione dei grafici direttamente all'interno delle celle del notebook.
%matplotlib inline

# pd.set_option("display.max_columns", None): Istruisce pandas a mostrare tutte le 33 colonne
# quando stampiamo a schermo il DataFrame, senza nascondere colonne con puntini di sospensione (...)
pd.set_option("display.max_columns", 50)

print("Librerie importate e ambiente configurato con successo!")

---
## 1.1 Caricamento e Ispezione Iniziale del Dataset

### Perché facciamo questo passaggio?
Prima di iniziare qualsiasi analisi statistica o di data mining, è obbligatorio verificare la conformità tecnica del dataset grezzo:
1. **Caricamento corretto**: Verificare che il file venga suddiviso nel corretto numero di righe e colonne specificando il delimitatore opportuno (`sep=';'`).
2. **Forma della matrice (`shape`)**: Misurare quante osservazioni (righe = venditori/dealer) e quanti attributi (colonne) sono presenti.
3. **Verifica della chiave primaria (`sp_id`)**: Accertarsi che ciascuna riga rappresenti un venditore unico senza duplicazioni accidentali (`nunique() == len(df)`).
4. **Ispezione dei tipi grezzi (`dtypes`)**: Verificare quali colonne sono state interpretate come numeri e quali come stringhe (`object`), per individuare subito problemi di parsing.

In [ ]:
# ==============================================================================
# CARICAMENTO DEL DATASET GREZZO
# ==============================================================================
# Definiamo il percorso al file CSV in modo robusto, funzionante sia che il notebook
# venga eseguito dalla cartella "notebooks/" sia dalla cartella principale del progetto.
if os.path.exists("../data/seller_info_dm1_2627.csv"):
    dataset_path = "../data/seller_info_dm1_2627.csv"
elif os.path.exists("data/seller_info_dm1_2627.csv"):
    dataset_path = "data/seller_info_dm1_2627.csv"
else:
    raise FileNotFoundError("Impossibile trovare il file 'seller_info_dm1_2627.csv' nella cartella data/")

# Specifichiamo sep=";" perché il file utilizza il punto e virgola come separatore di campo.
df = pd.read_csv(dataset_path, sep=";")

# df.shape restituisce una tupla (n_righe, n_colonne):
# - righe: unità statistiche (venditori / concessionari)
# - colonne: variabili osservate
n_righe, n_colonne = df.shape
print(f"Dataset caricato da: {dataset_path}")
print(f"Il dataset contiene {n_righe:,} record (venditori) e {n_colonne} attributi (colonne).\n")

# df.head(5): Mostra le prime 5 righe del DataFrame per un riscontro visivo immediato
df.head()

In [ ]:
# ==============================================================================
# VERIFICA DELLA CHIAVE PRIMARIA (sp_id) E DUPLICATI COMPLETI
# ==============================================================================
# In base alla documentazione del dataset, 'sp_id' (Seller Profile ID) rappresenta
# l'identificativo univoco di ciascun venditore aggregato.
# Eseguiamo due controlli fondamentali di Data Integrity:
# 1. Unicità di sp_id: ogni record deve fare riferimento a un venditore distinto.
# 2. Duplicati esatti: nessuna riga deve essere interamente identica a un'altra.

totale_righe = len(df)
sp_id_unici = df["sp_id"].nunique()
righe_duplicate_esatte = df.duplicated().sum()

print(f"Totale record nel dataset: {totale_righe}")
print(f"Valori distinti di 'sp_id': {sp_id_unici}")
print(f"Righe duplicate al 100%:    {righe_duplicate_esatte}")

if totale_righe == sp_id_unici and righe_duplicate_esatte == 0:
    print("\n[ESITO POSITIVO]: 'sp_id' è una chiave primaria valida ed è univoco. Non ci sono righe duplicate.")
else:
    print("\n[ATTENZIONE]: Rilevati duplicati nell'identificatore o nel dataset!")

In [ ]:
# ==============================================================================
# AUDIT DEI TIPI INFORMATICI RILEVATI DA PANDAS (dtypes) E VALORI MANCANTI
# ==============================================================================
# Creiamo una tabella sintetica che analizza per ciascuna delle 33 colonne:
# - Il tipo informatico assegnato automaticamente (dtype)
# - Il numero di record non nulli e di valori mancanti (NaN)
# - La percentuale di valori mancanti su 13.843 righe
# - Un valore di esempio per verificare visivamente il formato dei dati

audit_tipi = pd.DataFrame({
    "Attributo": df.columns,
    "Tipo Informatico (dtype)": df.dtypes.values,
    "Valori Non-Null": df.notnull().sum().values,
    "Valori Nulli (NaN)": df.isnull().sum().values,
    "% Mancanti": (df.isnull().mean() * 100).round(2).values,
    "Esempio Valore (Riga 0)": df.iloc[0].values
})

# Visualizziamo la tabella di audit
audit_tipi

### Rilevamento di un'anomalia di formato (Syntactic Data Quality)
Dall'ispezione della tabella sopra emerge subito un fatto rilevante per la qualità dei dati:
* Diverse colonne che descrivono grandezze fisiche continue (come `price_median`, `back_legroom_median`, `front_legroom_median`, `fuel_tank_volume_median`, `height_median`, `length_median`, `width_median`) risultano di tipo **`object` (stringhe)** anziché `float64`.
* **Causa**: Nel file CSV alcuni valori presentano stringhe come `'38.650.000.000.000.000'` o `'26.037.495.000.000.000'`, dovute a un problema di esportazione software (dove il punto è stato interpretato come separatore delle migliaia anziché decimale).
* **Nota metodologica**: Questo riscontro è da evidenziare esplicitamente nel Report sotto la voce *Syntactic Accuracy* e verrà corretto nella successiva fase di Data Preparation.

---
## 1.2 Data Semantics (Dizionario Dati e Classificazione delle Variabili)

### Fondamenti Teorici: Le Scale di Misura (Classificazione di Stevens)
Nel Data Mining e nella statistica descrittiva, il significato e le operazioni matematiche lecite dipendono strettamente dalla **scala di misura** dell'attributo:

1. **Nominale (Qualitativa pura)**: Le modalità sono etichette prive di qualsiasi ordine naturale (es. tipo di carrozzeria `dominant_body_type`, colore `dominant_listing_color`, tipo di carburante `dominant_fuel_type`).
   * *Operazioni consentite*: Uguaglianza / Diversità ($=, \neq$).
   * *Statistica di tendenza centrale*: **Moda** (la frequenza relativa delle classi).
2. **Ordinale (Qualitativa ordinabile)**: Le modalità possiedono una chiara graduatoria o gerarchia, ma la distanza tra livelli successivi non è misurabile o costante.
   * *Nel nostro dataset*: La variabile target **`seller_rating_class`** possiede un ordinamento naturale intrinseco: $\text{low} < \text{medium} < \text{high}$.
   * *Operazioni consentite*: Operatori di confronto ($<, >, \le, \ge$).
   * *Statistica di tendenza centrale*: **Mediana** e percentili, oltre alla moda.
3. **Intervallo (Quantitativa con zero convenzionale)**: I valori sono numerici e le differenze hanno un significato preciso, ma lo zero è arbitrario (non indica l'assenza della quantità).
   * *Nel nostro dataset*: L'anno di immatricolazione **`year_median`** (il calendario ha un anno zero convenzionale; dire "anno 2000" non significa il doppio dell'"anno 1000").
   * *Operazioni consentite*: Addizione e sottrazione (differenze di anni).
4. **Ratio / Rapporto (Quantitativa con zero assoluto)**: Esiste un punto zero fisico non arbitrario che rappresenta la totale assenza della grandezza misurata.
   * *Nel nostro dataset*: La maggior parte delle variabili numeriche: prezzi (`price_median`), chilometraggi (`mileage_median`), potenza motore (`horsepower_median`), dimensioni del veicolo, percentuali (`pct_*`), conteggi di giorni e annunci.
   * *Operazioni consentite*: Tutte le operazioni matematiche (rapporti, moltiplicazioni, medie, varianze, deviazioni standard).

### Ruolo delle Variabili nel Mining
* **Identificatore (ID)**: `sp_id` (chiave tecnica del record, da escludere dai modelli predittivi e dal clustering).
* **Target di Classificazione**: `seller_rating_class` (la classe di valutazione del venditore da predire nel modulo di classificazione).
* **Predittori / Feature**: Tutte le restanti 31 variabili che descrivono l'attività del venditore e la composizione del suo parco veicoli.

In [ ]:
# ==============================================================================
# COSTRUZIONE DELLA TABELLA SEMANTICA COMPLETA DEL DATASET
# ==============================================================================
# Definiamo in una struttura dati completa e trasparente il metadato di ogni colonna,
# riprendendo le definizioni ufficiali del file seller_level_vehicle_dataset_description.txt
# e classificando ciascuna secondo la teoria del corso.

dati_semantica = [
    # --- IDENTIFICATIVO ---
    {"Attributo": "sp_id", "Gruppo": "Identificativo", 
     "Descrizione": "Identificativo univoco del rivenditore/concessionario (Seller Profile ID).",
     "Scala Misura": "Nominale", "Tipo Dato": "Discreto (ID)", "Ruolo": "ID (Escluso dai modelli)", "Dominio/Unità": "Codice numerico univoco"},
    
    # --- ATTIVITÀ DEL VENDITORE E INVENTARIO ---
    {"Attributo": "n_listings", "Gruppo": "Attività & Volume", 
     "Descrizione": "Numero totale di annunci di veicoli associati al venditore.",
     "Scala Misura": "Ratio", "Tipo Dato": "Discreto (Conteggio)", "Ruolo": "Predittore", "Dominio/Unità": "Interi >= 1"},
    {"Attributo": "n_listing_days", "Gruppo": "Attività & Volume", 
     "Descrizione": "Numero di giorni di calendario distinti in cui sono stati osservati annunci del venditore.",
     "Scala Misura": "Ratio", "Tipo Dato": "Discreto (Conteggio)", "Ruolo": "Predittore", "Dominio/Unità": "Giorni (>= 1)"},
    {"Attributo": "listing_span_days", "Gruppo": "Attività & Volume", 
     "Descrizione": "Numero di giorni trascorsi tra il primo e l'ultimo annuncio osservato per il venditore.",
     "Scala Misura": "Ratio", "Tipo Dato": "Discreto", "Ruolo": "Predittore", "Dominio/Unità": "Giorni (>= 0)"},
    {"Attributo": "daysonmarket_median", "Gruppo": "Attività & Volume", 
     "Descrizione": "Mediana dei giorni di permanenza degli annunci di veicoli del venditore sul mercato.",
     "Scala Misura": "Ratio", "Tipo Dato": "Continuo", "Ruolo": "Predittore", "Dominio/Unità": "Giorni (>= 0)"},
    
    # --- PREZZO E RISPARMIO PUBBLICIZZATO ---
    {"Attributo": "price_median", "Gruppo": "Economico & Prezzo", 
     "Descrizione": "Prezzo mediano di vendita pubblicizzato per i veicoli del venditore.",
     "Scala Misura": "Ratio", "Tipo Dato": "Continuo", "Ruolo": "Predittore", "Dominio/Unità": "USD ($)"},
    {"Attributo": "savings_amount_median", "Gruppo": "Economico & Prezzo", 
     "Descrizione": "Importo mediano del risparmio/sconto pubblicizzato rispetto al valore di listino stimato.",
     "Scala Misura": "Ratio", "Tipo Dato": "Continuo", "Ruolo": "Predittore", "Dominio/Unità": "USD ($)"},
    
    # --- ANZIANITÀ E UTILIZZO DEI VEICOLI ---
    {"Attributo": "year_median", "Gruppo": "Veicolo: Storico", 
     "Descrizione": "Anno di fabbricazione/modello mediano dei veicoli offerti dal venditore.",
     "Scala Misura": "Intervallo", "Tipo Dato": "Continuo/Discreto", "Ruolo": "Predittore", "Dominio/Unità": "Anno di calendario (es. 2015)"},
    {"Attributo": "vehicle_age_median", "Gruppo": "Veicolo: Storico", 
     "Descrizione": "Età mediana dei veicoli al momento dell'annuncio (calcolata come anno annuncio meno anno modello).",
     "Scala Misura": "Ratio", "Tipo Dato": "Continuo", "Ruolo": "Predittore", "Dominio/Unità": "Anni (>= 0)"},
    {"Attributo": "mileage_median", "Gruppo": "Veicolo: Storico", 
     "Descrizione": "Chilometraggio mediano percorso dai veicoli del venditore.",
     "Scala Misura": "Ratio", "Tipo Dato": "Continuo", "Ruolo": "Predittore", "Dominio/Unità": "Miglia (miles)"},
    {"Attributo": "owner_count_median", "Gruppo": "Veicolo: Storico", 
     "Descrizione": "Numero mediano di precedenti proprietari dei veicoli del venditore.",
     "Scala Misura": "Ratio", "Tipo Dato": "Continuo", "Ruolo": "Predittore", "Dominio/Unità": "Conteggio proprietari (>= 0)"},
    
    # --- SPECIFICHE MOTORE E CONSUMI ---
    {"Attributo": "horsepower_median", "Gruppo": "Veicolo: Meccanica", 
     "Descrizione": "Potenza mediana del motore dei veicoli del venditore.",
     "Scala Misura": "Ratio", "Tipo Dato": "Continuo", "Ruolo": "Predittore", "Dominio/Unità": "Cavalli vapore (HP)"},
    {"Attributo": "engine_displacement_median", "Gruppo": "Veicolo: Meccanica", 
     "Descrizione": "Cilindrata mediana del motore dei veicoli del venditore.",
     "Scala Misura": "Ratio", "Tipo Dato": "Continuo", "Ruolo": "Predittore", "Dominio/Unità": "Centimetri cubi (cc)"},
    {"Attributo": "city_fuel_economy_median", "Gruppo": "Veicolo: Consumi", 
     "Descrizione": "Efficienza di consumo urbano mediana dei veicoli del venditore.",
     "Scala Misura": "Ratio", "Tipo Dato": "Continuo", "Ruolo": "Predittore", "Dominio/Unità": "MPG (Miles Per Gallon)"},
    {"Attributo": "highway_fuel_economy_median", "Gruppo": "Veicolo: Consumi", 
     "Descrizione": "Efficienza di consumo autostradale mediana dei veicoli del venditore.",
     "Scala Misura": "Ratio", "Tipo Dato": "Continuo", "Ruolo": "Predittore", "Dominio/Unità": "MPG (Miles Per Gallon)"},
    {"Attributo": "fuel_tank_volume_median", "Gruppo": "Veicolo: Consumi", 
     "Descrizione": "Capacità volumetrica mediana del serbatoio carburante.",
     "Scala Misura": "Ratio", "Tipo Dato": "Continuo", "Ruolo": "Predittore", "Dominio/Unità": "Galloni (gal)"},
    
    # --- DIMENSIONI FISICHE DEI VEICOLI ---
    {"Attributo": "length_median", "Gruppo": "Veicolo: Dimensioni", 
     "Descrizione": "Lunghezza mediana dei veicoli presenti nell'inventario del venditore.",
     "Scala Misura": "Ratio", "Tipo Dato": "Continuo", "Ruolo": "Predittore", "Dominio/Unità": "Pollici (inches)"},
    {"Attributo": "width_median", "Gruppo": "Veicolo: Dimensioni", 
     "Descrizione": "Larghezza mediana dei veicoli presenti nell'inventario del venditore.",
     "Scala Misura": "Ratio", "Tipo Dato": "Continuo", "Ruolo": "Predittore", "Dominio/Unità": "Pollici (inches)"},
    {"Attributo": "height_median", "Gruppo": "Veicolo: Dimensioni", 
     "Descrizione": "Altezza mediana dei veicoli presenti nell'inventario del venditore.",
     "Scala Misura": "Ratio", "Tipo Dato": "Continuo", "Ruolo": "Predittore", "Dominio/Unità": "Pollici (inches)"},
    {"Attributo": "front_legroom_median", "Gruppo": "Veicolo: Dimensioni", 
     "Descrizione": "Spazio mediano per le gambe nei sedili anteriori (abitabilità frontale).",
     "Scala Misura": "Ratio", "Tipo Dato": "Continuo", "Ruolo": "Predittore", "Dominio/Unità": "Pollici (inches)"},
    {"Attributo": "back_legroom_median", "Gruppo": "Veicolo: Dimensioni", 
     "Descrizione": "Spazio mediano per le gambe nei sedili posteriori (abitabilità posteriore).",
     "Scala Misura": "Ratio", "Tipo Dato": "Continuo", "Ruolo": "Predittore", "Dominio/Unità": "Pollici (inches)"},
    
    # --- COMPOSIZIONE DEL PARCO VEICOLI (PROPORZIONI) ---
    {"Attributo": "pct_isCab", "Gruppo": "Composizione Flotta", 
     "Descrizione": "Proporzione di veicoli precedentemente utilizzati come taxi o veicoli a noleggio.",
     "Scala Misura": "Ratio", "Tipo Dato": "Continuo", "Ruolo": "Predittore", "Dominio/Unità": "Proporzione [0.0, 1.0]"},
    {"Attributo": "pct_is_new", "Gruppo": "Composizione Flotta", 
     "Descrizione": "Proporzione di veicoli classificati come veicoli nuovi.",
     "Scala Misura": "Ratio", "Tipo Dato": "Continuo", "Ruolo": "Predittore", "Dominio/Unità": "Proporzione [0.0, 1.0]"},
    {"Attributo": "pct_salvage", "Gruppo": "Composizione Flotta", 
     "Descrizione": "Proporzione di veicoli con certificato 'salvage' (veicoli incidentati/ricostruiti).",
     "Scala Misura": "Ratio", "Tipo Dato": "Continuo", "Ruolo": "Predittore", "Dominio/Unità": "Proporzione [0.0, 1.0]"},
    {"Attributo": "pct_theft_title", "Gruppo": "Composizione Flotta", 
     "Descrizione": "Proporzione di veicoli con storico documentale legato a furto.",
     "Scala Misura": "Ratio", "Tipo Dato": "Continuo", "Ruolo": "Predittore", "Dominio/Unità": "Proporzione [0.0, 1.0]"},
    
    # --- CARATTERISTICHE MODALI/DOMINANTI DELL'INVENTARIO ---
    {"Attributo": "dominant_body_type", "Gruppo": "Caratteristiche Modali", 
     "Descrizione": "Tipologia di carrozzeria più frequente nell'inventario del venditore (moda).",
     "Scala Misura": "Nominale", "Tipo Dato": "Discreto (Testo)", "Ruolo": "Predittore", "Dominio/Unità": "suv_crossover, sedan, pickup..."},
    {"Attributo": "dominant_fuel_type", "Gruppo": "Caratteristiche Modali", 
     "Descrizione": "Tipo di alimentazione prevalente nell'inventario del venditore.",
     "Scala Misura": "Nominale", "Tipo Dato": "Discreto (Testo)", "Ruolo": "Predittore", "Dominio/Unità": "gasoline, diesel, hybrid, electric"},
    {"Attributo": "dominant_transmission", "Gruppo": "Caratteristiche Modali", 
     "Descrizione": "Tipo di trasmissione/cambio prevalente nei veicoli del venditore.",
     "Scala Misura": "Nominale", "Tipo Dato": "Discreto (Testo)", "Ruolo": "Predittore", "Dominio/Unità": "a (automatic), m (manual), cvt"},
    {"Attributo": "dominant_wheel_system", "Gruppo": "Caratteristiche Modali", 
     "Descrizione": "Configurazione di trazione prevalente (drivetrain) nell'inventario.",
     "Scala Misura": "Nominale", "Tipo Dato": "Discreto (Testo)", "Ruolo": "Predittore", "Dominio/Unità": "fwd, rwd, awd, 4wd"},
    {"Attributo": "dominant_listing_color", "Gruppo": "Caratteristiche Modali", 
     "Descrizione": "Colore esterno standardizzato più frequente nei veicoli del venditore.",
     "Scala Misura": "Nominale", "Tipo Dato": "Discreto (Testo)", "Ruolo": "Predittore", "Dominio/Unità": "black, white, silver, blue..."},
    
    # --- PROFILO AZIENDALE DEL RIVENDITORE ---
    {"Attributo": "franchise_status", "Gruppo": "Profilo Dealer", 
     "Descrizione": "Tipologia di concessionario (franchise = affiliato a un marchio, independent = multimarca).",
     "Scala Misura": "Nominale", "Tipo Dato": "Discreto (Testo)", "Ruolo": "Predittore", "Dominio/Unità": "franchise, independent, mixed, unknown"},
    {"Attributo": "new_used_profile", "Gruppo": "Profilo Dealer", 
     "Descrizione": "Composizione commerciale dell'inventario (vendita di auto nuove, usate o miste).",
     "Scala Misura": "Nominale", "Tipo Dato": "Discreto (Testo)", "Ruolo": "Predittore", "Dominio/Unità": "new_only, used_only, mixed, unknown"},
    
    # --- VARIABILE TARGET DI PREDIZIONE ---
    {"Attributo": "seller_rating_class", "Gruppo": "Target di Predizione", 
     "Descrizione": "Classe qualitativa di reputazione/rating del venditore (discretizzata in 3 fasce).",
     "Scala Misura": "Ordinale", "Tipo Dato": "Discreto (Testo)", "Ruolo": "TARGET", "Dominio/Unità": "low < medium < high"}
]

# Creiamo il DataFrame della Tabella Semantica
tabella_semantica = pd.DataFrame(dati_semantica)

# Visualizziamo la tabella completa
tabella_semantica

In [ ]:
# ==============================================================================
# RIEPILOGO STATISTICO DELLE VARIABILI PER SCALA DI MISURA E RUOLO
# ==============================================================================
# Verifichiamo come si ripartiscono le 33 colonne complessive del dataset

print("--- Ripartizione per Scala di Misura (Stevens) ---")
conteggio_scale = tabella_semantica["Scala Misura"].value_counts()
for scala, count in conteggio_scale.items():
    print(f"* {scala:<12}: {count:>2} variabili ({count/len(tabella_semantica)*100:.1f}%)")

print("\n--- Ripartizione per Ruolo nel Mining ---")
conteggio_ruoli = tabella_semantica["Ruolo"].value_counts()
for ruolo, count in conteggio_ruoli.items():
    print(f"* {ruolo:<25}: {count:>2} variabili")

### Ispezione Dettagliata delle Variabili Categoriche e del Target
Per tutte le variabili qualitative (nominali e ordinali), esaminiamo la cardinalità (numero di categorie distinte), le frequenze e la presenza di categorie speciali (come `'unknown'`, stringhe anomale o valori nulli `NaN`).

In [ ]:
# ==============================================================================
# ISPEZIONE DELLE CATEGORIE DISCRETE E DEL TARGET
# ==============================================================================
# Elenco delle variabili qualitative identificate nella tabella semantica:
colonne_categoriche = [
    "seller_rating_class",     # TARGET (Ordinale)
    "franchise_status",        # Profilo venditore
    "new_used_profile",        # Profilo inventario
    "dominant_body_type",      # Carrozzeria prevalente
    "dominant_fuel_type",      # Carburante prevalente
    "dominant_transmission",   # Cambio prevalente
    "dominant_wheel_system",   # Trazione prevalente
    "dominant_listing_color"   # Colore prevalente
]

print("==============================================================================")
print("FREQUENZE DELLE VARIABILI CATEGORICHE (CON CONTEGGIO ASSOLUTO E PERCENTUALE)")
print("==============================================================================\n")

for col in colonne_categoriche:
    print(f"---> ATTRIBUTO: {col} (Valori unici: {df[col].nunique(dropna=False)})")
    
    # Calcoliamo frequenze assolute (conteggio righe) e relative (percentuali sul totale)
    freq_assoluta = df[col].value_counts(dropna=False)
    freq_percentuale = (df[col].value_counts(dropna=False, normalize=True) * 100).round(2)
    
    # Combiniamo in una tabella comoda da leggere
    df_freq = pd.DataFrame({
        "Conteggio": freq_assoluta,
        "Percentuale (%)": freq_percentuale
    })
    display(df_freq)
    print("-" * 80)

### Prime Evidenze Empiriche per il Report (Sezioni 1.1 e 1.2)

Dall'analisi semantica e dalle frequenze emergono considerazioni fondamentali da discutere nel Report:

1. **Distribuzione del Target (`seller_rating_class`)**:
   * La classe modale è nettamente `medium` (~55%), seguita da `low` (~25%) e `high` (~20%).
   * Il dataset presenta quindi un target **moderatamente sbilanciato**, aspetto di cui dovremo tenere conto nella fase di modellazione (utilizzando metriche come *F1-score macro* o *matrice di confusione* anziché la semplice accuratezza).
2. **Presenza di Valori Mancanti Fisiologici e Categorie `unknown`**:
   * Nella colonna `dominant_wheel_system`, molti venditori presentano `NaN` (valore non registrato).
   * In variabili come `dominant_listing_color`, `franchise_status` e `new_used_profile`, è presente esplicitamente la stringa `'unknown'`: si tratta di un **"mancante mascherato"** (not missing at random ma codificato esplicitamente come categoria a sé stante).
3. **Trazione e Trasmissione**:
   * La trasmissione è quasi ovunque automatica (`a`), mentre le altre tipologie (`m`, `cvt`) sono classi rare.
4. **Prossimo Passo (Punto 1.3 - Data Quality & Preparation)**:
   * Procedere alla sanificazione sintattica dei valori con formattazione numerica alterata (punti multipli) per convertire le 7 colonne di dimensioni/prezzo da `object` a `float64`.
   * Eseguire i controlli di coerenza logica e vincoli di dominio (sanity checks).